In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

In [2]:
class DualEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(num_embeddings=vocab_size, embedding_dim=embed_dim)
        
        self.history_rnn = nn.LSTM(embed_dim, hid_dim, n_layers, dropout=dropout)
        self.document_rnn = nn.LSTM(embed_dim, hid_dim, n_layers, dropout=dropout)

        self.dropout = nn.Dropout(dropout)

    def forward(self, history, document):
        embed_hist = self.dropout(self.embed(history))
        embed_doc = self.dropout(self.embed(document))

        hist_output, (hidden_h, cell_h) = self.history_rnn(embed_hist)

        doc_output, (hidden_d, cell_d) = self.document_rnn(embed_doc)

        return hist_output, hidden_h, cell_h, doc_output, hidden_d, cell_d
        


In [5]:
class DualCoAttentionDecoder(nn.Module):
    def __init__(self, vocab_size, emb_dim, enc_hid_dim, dec_hid_dim, n_layers, dropout):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim)

        self.attention_hist = nn.Linear(enc_hid_dim + dec_hid_dim, dec_hid_dim)
        self.v_hist = nn.Linear(dec_hid_dim, 1, bias=False)

        self.attention_doc = nn.Linear(enc_hid_dim + dec_hid_dim, dec_hid_dim)
        self.v_doc = nn.Linear(dec_hid_dim, 1, bias = False)

        self.rnn = nn.LSTM(emb_dim + 2 * enc_hid_dim, dec_hid_dim, n_layers, dropout)

        self.fc_out = nn.Linear(emb_dim + dec_hid_dim + (enc_hid_dim)*2, vocab_size)

        self.dropout = nn.Dropout(dropout)
    
    def calc_attention(self, hidden, encoder_outputs, attn_layer, v_layer):
        batch_size = encoder_outputs.shape[1]
        src_len = encoder_outputs.shape[0]

        hidden = hidden.squeeze(0).unsqeeze(1).repeat(1, src_len, 1)

        encoder_outputs = encoder_outputs.permute(1, 0, 2)

        en = torch.tanh(attn_layer(torch.cat((hidden, encoder_outputs), dim = 2)))

        attention = v_layer(en).squeeze(2)

        return F.softmax(attention, dim=1)

    def forward(self, input, hidden, cell, hist_out, doc_out):
        input = input.unsqueeze(0)

        embedded = self.dropout(self.embedding(input))

        a_hist = self.calc_attention(hidden[-1:], hist_out, self.attn_hist, self.v_hist)
        c_hist = torch.bmm(a_hist.unsqueeze(1), hist_out.permute(1, 0, 2)).permute(1, 0, 2)
        
        a_doc = self.calc_attention(hidden[-1:], doc_out, self.attn_doc, self.v_doc)
        c_doc = torch.bmm(a_doc.unsqueeeze(1), hist_out.permute(1, 0, 2)).permute(1, 0, 2)

        rnn_input = torch.cat((embedded, c_hist, c_doc), dim = 2)
        output, (hidden, cell) = self.rnn(rnn_input, hidden, cell)

        predictions = self.fc_out(torch.cat((output.squeeze(0), c_hist.squeeze(0), c_doc.squeeze(0), embedded.squeeze(0)), dim=1))

        return predictions, hidden, cell
